# Compare CIRPIN DB to CPDB

In [1]:

# 1) Load CPDB tm score data

import numpy as np
import pandas as pd

fp = '/home/ubuntu/plmCP/CPDB/tm_scores_cpdb.txt'

df = pd.read_csv(fp, sep=' ')

# 2) Filter for real CPs

df_tm_cp_above05 = df[df['tmscore-CP'] > 0.5]

df_tm_cp_above05_diff_greater_than_0 = df_tm_cp_above05[(df_tm_cp_above05['tmscore-CP']-df_tm_cp_above05['tmscore']) > 0]

print(f'After filtering based on TM CP difference {len(df_tm_cp_above05_diff_greater_than_0)}')
df_tm_cp_above05_diff_greater_than_0.head()

cp_data = df_tm_cp_above05_diff_greater_than_0
cp_data = cp_data.rename(columns={
    'Protein1': 'Query',
    'Protein2': 'Target'
})

After filtering based on TM CP difference 3181


In [106]:
len(set(list(cp_data['Query']) + list(cp_data['Target'])))


1667

In [112]:
df[df['Protein2'] == '1pujA']

,Protein1,Protein2,tmscore,tmscore-CP
3239,121pA,1pujA,0.48763,0.71325
3244,1a2bA,1pujA,0.45876,0.66868
3246,1a4rA,1pujA,0.45159,0.65098
3247,1aa9A,1pujA,0.46076,0.67352
3251,1agpA,1pujA,0.48817,0.71270
...,...,...,...,...
3983,421pA,1pujA,0.48570,0.70463
3984,4q21A,1pujA,0.47863,0.69990
3986,521pA,1pujA,0.48858,0.70766
3989,621pA,1pujA,0.48488,0.70335


In [113]:
cp_data[cp_data['Target'] == '1PUJ_A']

,Query,Target,tmscore,tmscore-CP
3239,121P_A,1PUJ_A,0.48763,0.71325
3244,1A2B_A,1PUJ_A,0.45876,0.66868
3246,1A4R_A,1PUJ_A,0.45159,0.65098
3247,1AA9_A,1PUJ_A,0.46076,0.67352
3251,1AGP_A,1PUJ_A,0.48817,0.71270
...,...,...,...,...
3983,421P_A,1PUJ_A,0.48570,0.70463
3984,4Q21_A,1PUJ_A,0.47863,0.69990
3986,521P_A,1PUJ_A,0.48858,0.70766
3989,621P_A,1PUJ_A,0.48488,0.70335


In [2]:
# 3) Fix naming to match naming of downloaded PDBs

def convert_pdb_name_reverse(name):
    """Convert PDB name from '1xyzA' format to '1XYZ_A' format"""
    pdb_id = name[:4].upper()
    chain = name[4:]
    return f"{pdb_id}_{chain}"

# Apply to Query and Target columns
cp_data['Query'] = cp_data['Query'].apply(convert_pdb_name_reverse)
cp_data['Target'] = cp_data['Target'].apply(convert_pdb_name_reverse)

In [3]:
cp_data.head()

,Query,Target,tmscore,tmscore-CP
1,1FP3_A,1WU4_A,0.67768,0.67887
2,1FP3_A,2DRQ_A,0.67558,0.67691
3,1FP3_A,1WU6_A,0.67434,0.67558
4,1FP3_A,1HZF_A,0.54695,0.55763
5,1FP3_A,1KS8_A,0.63233,0.69125


In [31]:
# 4) Load foldseek clustering results of the CPDB
# All the PDBs in the above set were downloaded and then clustered using foldseek command below
# clustering done with:  foldseek easy-cluster . cluster_results tmp -c 0.9 -e 0.01

cluster_fp = 'cluster_results_clean_cluster.tsv'

fs_cluster = pd.read_csv(cluster_fp, sep='\t', header =None)
fs_cluster.columns = ['Cluster','Structure']

cluster_dict = (
    fs_cluster.groupby('Cluster')['Structure']
      .apply(list)
      .to_dict()
)

### create structure_to_cluster_dict
z = []
structure_to_cluster_dict = {}
for clust, strucs in cluster_dict.items():
    for struc in strucs:
        structure_to_cluster_dict[struc] = clust
print(f'Number of CPDB foldseek clusters: {len(cluster_dict)}')


Number of CPDB foldseek clusters: 400


In [36]:
cluster_sizes = {}
for clust, strucs in cluster_dict.items():
    clust_size = len(strucs)
    cluster_sizes[clust] = clust_size

In [40]:
size_counts = {}

for size in cluster_sizes.values():
    size_counts[size] = size_counts.get(size, 0) + 1

print(size_counts)

{13: 1, 1: 219, 3: 36, 12: 3, 4: 14, 6: 7, 2: 63, 9: 5, 10: 2, 34: 1, 5: 9, 16: 1, 18: 2, 7: 9, 46: 2, 8: 5, 23: 1, 24: 2, 35: 1, 32: 2, 30: 1, 15: 2, 20: 1, 40: 2, 94: 1, 65: 1, 26: 2, 37: 1, 17: 1, 43: 1, 22: 1, 14: 1}


### Check the overlap between the cpdb_pdbs and the names in the foldseek cluster results

In [6]:
### Get list of pdbs in the dir:
import os

cpdb_pdbs = [f[:-4] for f in os.listdir() if f.endswith('.pdb')]
pdb_count_in_dir = len(cpdb_pdbs)
print(f"Number of .pdb files: {pdb_count_in_dir}")



## Get number of unique pdbs in the cp_data dataframe above
foldseek_structures = list(fs_cluster['Structure'])
a =[]
for i in foldseek_structures:
    a.append(i)
    
print(f'Number of .pdbs in Foldseek clustering: {len(set(a))}')

if len(set(a)) != pdb_count_in_dir:
    print(f'Error! Discrepancy of {len(set(a))-pdb_count_in_dir}')
    pdbs_foldseek_created = []
    for i in foldseek_structures:
        if i not in cpdb_pdbs:
            pdbs_foldseek_created.append(i)

            
fs_missing = []
if len(set(a)) != pdb_count_in_dir:
    print(f'Error! Discrepancy of {len(set(a))-pdb_count_in_dir}')
    pdbs_foldseek_created = []
    for i in cpdb_pdbs:
        if i not in a:
            fs_missing.append(i)
print(fs_missing)

Number of .pdb files: 1665
Number of .pdbs in Foldseek clustering: 1665
[]


#### Removed 1QTJ_A and 2PGK_A because they lacked sequence (not detected by Foldseek)

In [42]:
    
def check_cluster(cluster, cluster_dict, structure_to_cluster_dict, cp_data):
    '''Check if cluster can be merged with a different cluster 
    Input: Rep structure of the cluster (cluster_dict key)
    Returns: List of tuples containing (current_cluster, other_cluster, connecting_structure)
    '''
    structures_in_cluster = cluster_dict[cluster]
    merge_candidates = []
    
    for struc in structures_in_cluster:
        # Filter cp_data for rows that have struc in either query or target
        # These selected rows are CP connections to other structures
        cp_connections = cp_data[
            (cp_data['Query'] == struc) |
            (cp_data['Target'] == struc)
        ]
        
        # Get list of names appearing in Query or Target col that are NOT struc
        other_structures = set()
        other_structures.update(cp_connections['Query'].values)
        other_structures.update(cp_connections['Target'].values)
        other_structures.discard(struc)  # Remove struc itself
        
        # For each name, check which cluster it belongs to
        for other_struc in other_structures:
            if other_struc in structure_to_cluster_dict:
                other_cluster = structure_to_cluster_dict[other_struc]
                
                # If it belongs to a DIFFERENT cluster, these two clusters can be merged
                if other_cluster != cluster:
                    #print('DIFFERENT')
                    merge_candidates.append((cluster, other_cluster))
    ## If no other clusters could be merged, just record the cluster itself
    if len(merge_candidates) == 0:
        merge_candidates.append((cluster, cluster))
    return merge_candidates

In [43]:
# 5) Create merged clusters
cluster_mergers = []
for c in (cluster_dict.keys()):
    results = check_cluster(c, cluster_dict,structure_to_cluster_dict,cp_data)
    cluster_mergers.append(results)


In [64]:
# Get unique mergers
cluster_mergers_unique = []

for i in cluster_mergers:
    for x in i:
        a = list(x)
        if a not in cluster_mergers_unique:
            cluster_mergers_unique.append(a)

In [86]:
# Create graph
import networkx as nx

G = nx.Graph() 
G.add_nodes_from(cluster_dict.keys())
G.add_edges_from(cluster_mergers_unique)
    
# Find connected components
components = list(nx.connected_components(G))

In [102]:
len(components)

69

In [89]:
merged_clusters = {}
    
for i, component in enumerate(components):
    # Create name for merged cluster
    component_list = sorted(list(component))
    merged_name = '_'.join(component_list)
    
    # Combine all structures from clusters in this component
    merged_structures = []
    for cluster_name in component:
        merged_structures.extend(cluster_dict[cluster_name])
    
    # Remove duplicates while preserving order
    #merged_structures = list(dict.fromkeys(merged_structures))
    
    merged_clusters[merged_name] = merged_structures

In [98]:
len(merged_clusters)

69

In [100]:
cp_data[cp_data['Query'] == '1UCP_A']

,Query,Target,tmscore,tmscore-CP
849,1UCP_A,1AK4_C,0.43907,0.52235


### Embedding proteins
-----------------------------------------------------------------

In [ ]:
import numpy as np
import pandas as pd

fp = '/home/ubuntu/plmCP/CPDB/tm_scores_cpdb.txt'

df = pd.read_csv(fp, sep=' ')

In [ ]:
import progres as pg
import os
import torch

model_setting = 'Progres'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if model_setting == 'CIRPIN':
    cirpin = '/home/ubuntu/CIRPIN/trained_models/CIRPIN_model/CIRPIN_model_5k_cp_epoch301.pt'
    loaded_trained_model = pg.load_trained_model(device=device, trained_model=cirpin)
else:
    loaded_trained_model =pg.load_trained_model(device=device)
    


if 'Progres' not in df.columns:
    df['Progres'] = None


for idx, row in df.iterrows():
    pdb1 = row.Protein1
    pdb1 = pdb1[:4].upper() + '_' + pdb1[4].upper() + '.pdb'
    pdb2 = row.Protein2
    pdb2 = pdb2[:4].upper() + '_' + pdb2[4].upper() + '.pdb'

    cpdb_pdbs = '/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2'

    if pdb1 in os.listdir(cpdb_pdbs) and pdb2 in os.listdir(cpdb_pdbs):
        emb1 = pg.embed_structure(querystructure=pdb1, device=device, model=loaded_trained_model)
        emb2 = pg.embed_structure(querystructure=pdb2, device=device, model=loaded_trained_model)
        cosine_dist = (emb1 * emb2).sum(dim=-1) # Normalised in the model --> Need to normalize for the node level embeddings since this is NOT normalized in the model
        score = (1 + cosine_dist) / 2
        df.loc[row.name, 'Progres'] = score.item()
        print(f'{pdb1}, {pdb2}, {score}')


In [ ]:
df.head()

In [ ]:
df['prog_cirpin_diff'] = df['CIRPIN'] - df['Progres']
df['tmscore_diff'] = df['tmscore-CP'] - df['tmscore']

df_sorted = df.sort_values(by='prog_cirpin_diff', ascending=False)




In [ ]:
df_sorted = df.sort_values(by='tmscore_diff', ascending=False)


In [ ]:
for idx, row in df_sorted.head(100).iterrows():
    print(row)

In [ ]:
df

In [ ]:
# Plot histogram
import matplotlib.pyplot as plt

plt.hist(df['prog_cirpin_diff'], bins=50, edgecolor='black')  # adjust bins as needed
plt.xlabel('Scores')
plt.ylabel('Frequency')
plt.title('Distribution of Scores')
plt.show()

In [ ]:
# Create scatter plot
plt.scatter(df['tmscore_diff'], df['prog_cirpin_diff'])

plt.xlabel('X Values')
plt.ylabel('Y Values')
plt.title('Scatter Plot')
plt.show()

In [ ]:
plt.hist(df['tmscore_diff'], bins=50, edgecolor='black')  # adjust bins as needed
plt.xlabel('Scores')
plt.ylabel('Frequency')
plt.title('Distribution of Scores')
plt.show()

In [ ]:
df.head()


In [ ]:
for row in df.iterrows():
    pdb1 = concat(row['Protein1'].split('_')[0].upper(), row['Protein1'].split('_')[1]).upper()
    pdb2 = concat(row['Protein2'].split('_')[0].upper(), row['Protein2'].split('_')[1]).upper()
    if pdb1 in os.listdir(cpdb_pdbs) and pdb2 in os.listdir(cpdb_pdbs):
        emb1 = pg.embed_structure(querystructure=pdb1, device=device,model=loaded_trained_model)
        emb2 = pg.embed_structure(querystructure=pdb2, device=device,model=loaded_trained_model)
        df.loc[row.name, 'TM-score'] = tmscore(emb1, emb2)


In [ ]:
import os 



for pdb in os.listdir(cpdb_pdbs):
    if pdb.endswith('.pdb'):
        pdb1 = pdb
        emb1 = pg.embed_structure(querystructure=pdb1, device=device,model=loaded_trained_model)